In [1]:
import json
from monty.json import MontyDecoder
import os
from pymatgen.core import Element

from utils_kga.statistical_analysis.analyze_ion_pairs import *

In [2]:
# Load edge-df
with open("data_and_plots/dfs_of_magnetic_edge_information_0p5_zero-magmom_threshold.json") as f:
    dict_all_stats = json.load(f)
all_stats = {key: pd.DataFrame.from_dict(df) for key, df in dict_all_stats.items()}

# For metadata filtering and computation of occurrences in magnetic primitive cells
with open("data_and_plots/df_stable_and_unique_MP_db.json") as f:
    df = json.load(f, cls=MontyDecoder)

plot_dir = "data_and_plots"

In [3]:
# The automatically determined oxidation states might be "unphysical" in many cases, e.g., that some ions are not expected to display magnetic order
# Nevertheless, we are showing the results as determined by the automated BVA
custom_sort_by_electron_configuration = [ 
    ("W", 6),
    ("V", 4),  ("Cr", 5), ("Nb", 4), ("Mo", 5), ("W", 5),
    ("V", 3), ("Cr", 4), ("Mn", 5), ("Mo", 4), ("W", 4),  ("Re", 5), ("Os", 6),
    ("V", 2), ("Cr", 3), ("Mn", 4), ("Ru", 5), ("Mo", 3), ("Os", 5),
    ("Cr", 2), ("Mn", 3), ("Fe", 4),
    ("Ru", 4), ("Ir", 5),
    ("Mn", 2), ("Fe", 3), ("Co", 4),
    ("Rh", 4), ("Ir", 4),
    ("Fe", 2), ("Co", 3), ("Ni", 4),
    ("Fe", 1), ("Co", 2), ("Ni", 3),
    ("Co", 1), ("Ni", 2),
    ("Ni", 1), ("Cu", 2)
]

In [4]:
for ang_df in all_stats.values():
    ang_df["site_is_tm"] = ang_df["site_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["site_to_is_tm"] = ang_df["site_to_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["ligand_el_set"] = ang_df["ligand_elements"].apply(lambda ls: set(ls))


In [5]:
write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

missed_ions = set()

data_string = f"corner connections with TM octahedra at both nodes"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[subdf[f"connectivity"]=="corner"]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]

    if not subdf.empty:
        n_lattice_points = df.at[md_id, "n_mag_lattice_points"]
        for mag_type, condition in zip(["afm", "fm", ],
                                        [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
            type_df = subdf.loc[condition]
            if not type_df.empty:
                abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                for k, v in abs_ion_pairs.items():
                    # Assert no detected ions are missing
                    try:
                        assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                        assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                    except AssertionError as e:
                        missed_ions.add(str(e))
                    if k in occus[mag_type]:
                        occus[mag_type][k] += v
                    else:
                        occus[mag_type][k] = v

                    if k in num_structures:
                        num_structures[k].add(md_id)
                    else:
                        num_structures[k] = {md_id}



print(occus)


# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
ratio_fig.update_xaxes(dict(
        tickfont=dict(size=15)))
ratio_fig.update_yaxes(dict(
    tickfont=dict(size=15)))

# ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))


n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))
print("missed: ", missed_ions)

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, 
                                         log=False, ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
numstructfig.update_xaxes(dict(
        tickfont=dict(size=15)))
numstructfig.update_yaxes(dict(
    tickfont=dict(size=15)))
# numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

corner connections with TM octahedra at both nodes
{'afm': {('Fe', 3, 'Fe', 3): 512.0, ('Fe', 1, 'Fe', 1): 36.0, ('Cr', 3, 'Cr', 3): 540.0, ('Fe', 2, 'Fe', 2): 228.0, ('V', 3, 'V', 3): 324.0, ('Mn', 4, 'Mn', 4): 244.0, ('Fe', 3, 'Mo', 5): 12.0, ('Mo', 5, 'Fe', 3): 12.0, ('Mn', 2, 'Mn', 2): 784.0, ('Co', 2, 'Co', 2): 310.0, ('Co', 4, 'Co', 4): 36.0, ('Ni', 2, 'Ni', 2): 464.0, ('Mn', 4, 'Co', 2): 36.0, ('Co', 2, 'Mn', 4): 36.0, ('Cr', 4, 'Mo', 4): 16.0, ('Mo', 4, 'Cr', 4): 16.0, ('Ni', 2, 'Os', 6): 24.0, ('Os', 6, 'Ni', 2): 24.0, ('Mn', 3, 'Cr', 3): 28.0, ('Cr', 3, 'Mn', 3): 28.0, ('Mn', 3, 'Mn', 3): 184.0, ('Fe', 3, 'Os', 5): 16.0, ('Os', 5, 'Fe', 3): 16.0, ('Mn', 3, 'Mn', 2): 64.0, ('Mn', 2, 'Mn', 3): 64.0, ('Mn', 2, 'Fe', 3): 16.0, ('Fe', 3, 'Mn', 2): 16.0, ('Co', 2, 'Co', 3): 84.0, ('Co', 3, 'Co', 3): 160.0, ('Co', 3, 'Co', 2): 84.0, ('V', 3, 'V', 4): 132.0, ('V', 4, 'V', 3): 132.0, ('Ni', 3, 'Ni', 3): 32.0, ('V', 2, 'V', 2): 108.0, ('Cu', 2, 'Ni', 2): 24.0, ('Ni', 2, 'Cu', 2): 24.0,

In [6]:
# Repeat analysis considering only oxygen edges

write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

missed_ions = set()

data_string = f"corner connections with TM octahedra at both nodes only oxygen edges"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[subdf[f"connectivity"]=="corner"]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]
    subdf = subdf.loc[subdf["ligand_el_set"]=={"O"}]

    if not subdf.empty:
        n_lattice_points = df.at[md_id, "n_mag_lattice_points"]
        for mag_type, condition in zip(["afm", "fm", ],
                                        [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
            type_df = subdf.loc[condition]
            if not type_df.empty:
                abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                for k, v in abs_ion_pairs.items():
                    # Assert no detected ions are missing
                    try:
                        assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                        assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                    except AssertionError as e:
                        missed_ions.add(str(e))
                    if k in occus[mag_type]:
                        occus[mag_type][k] += v
                    else:
                        occus[mag_type][k] = v

                    if k in num_structures:
                        num_structures[k].add(md_id)
                    else:
                        num_structures[k] = {md_id}



print(occus)

# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
ratio_fig.update_xaxes(dict(
        tickfont=dict(size=15)))
ratio_fig.update_yaxes(dict(
    tickfont=dict(size=15)))

# ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))


n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))
print("missed: ", missed_ions)

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, 
                                         log=False, ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
numstructfig.update_xaxes(dict(
        tickfont=dict(size=15)))
numstructfig.update_yaxes(dict(
    tickfont=dict(size=15)))
# numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

corner connections with TM octahedra at both nodes only oxygen edges
{'afm': {('Fe', 3, 'Fe', 3): 408.0, ('Fe', 1, 'Fe', 1): 36.0, ('Cr', 3, 'Cr', 3): 516.0, ('Fe', 2, 'Fe', 2): 168.0, ('V', 3, 'V', 3): 270.0, ('Mn', 4, 'Mn', 4): 244.0, ('Fe', 3, 'Mo', 5): 12.0, ('Mo', 5, 'Fe', 3): 12.0, ('Mn', 2, 'Mn', 2): 684.0, ('Co', 2, 'Co', 2): 286.0, ('Co', 4, 'Co', 4): 36.0, ('Ni', 2, 'Ni', 2): 404.0, ('Mn', 4, 'Co', 2): 36.0, ('Co', 2, 'Mn', 4): 36.0, ('Cr', 4, 'Mo', 4): 16.0, ('Mo', 4, 'Cr', 4): 16.0, ('Ni', 2, 'Os', 6): 24.0, ('Os', 6, 'Ni', 2): 24.0, ('Mn', 3, 'Cr', 3): 28.0, ('Cr', 3, 'Mn', 3): 28.0, ('Mn', 3, 'Mn', 3): 148.0, ('Fe', 3, 'Os', 5): 16.0, ('Os', 5, 'Fe', 3): 16.0, ('Mn', 3, 'Mn', 2): 48.0, ('Mn', 2, 'Mn', 3): 48.0, ('Co', 2, 'Co', 3): 48.0, ('Co', 3, 'Co', 3): 152.0, ('Co', 3, 'Co', 2): 48.0, ('V', 3, 'V', 4): 116.0, ('V', 4, 'V', 3): 116.0, ('Ni', 3, 'Ni', 3): 32.0, ('V', 2, 'V', 2): 104.0, ('Cu', 2, 'Ni', 2): 24.0, ('Ni', 2, 'Cu', 2): 24.0, ('V', 4, 'V', 4): 96.0, ('Fe', 3,

In [7]:
# Repeat analysis only considering structures with one magnetic ion to eliminate possible mixed valence issue a bit

write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

missed_ions = set()

data_string = f"corner connections with TM octahedra at both nodes only one mag. ion per structure"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    if len(set(ang_df["site_element"])) == 1 and len(set(ang_df["site_oxidation"])) == 1:
        subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
        subdf = subdf.loc[subdf[f"connectivity"]=="corner"]
        subdf = subdf.loc[(subdf["site_ce"]=="O:6")
                & (subdf["site_to_ce"]=="O:6")]

        if not subdf.empty:
            n_lattice_points = df.at[md_id, "n_mag_lattice_points"]
            for mag_type, condition in zip(["afm", "fm", ],
                                            [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
                type_df = subdf.loc[condition]
                if not type_df.empty:
                    abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                    for k, v in abs_ion_pairs.items():
                        # Assert no detected ions are missing
                        try:
                            assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                            assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                        except AssertionError as e:
                            missed_ions.add(str(e))
                        if k in occus[mag_type]:
                            occus[mag_type][k] += v
                        else:
                            occus[mag_type][k] = v

                        if k in num_structures:
                            num_structures[k].add(md_id)
                        else:
                            num_structures[k] = {md_id}


# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
ratio_fig.update_xaxes(dict(
        tickfont=dict(size=15)))
ratio_fig.update_yaxes(dict(
    tickfont=dict(size=15)))

# ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))


n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))
print("missed: ", missed_ions)

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, 
                                         log=False, ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
numstructfig.update_xaxes(dict(
        tickfont=dict(size=15)))
numstructfig.update_yaxes(dict(
    tickfont=dict(size=15)))
# numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

corner connections with TM octahedra at both nodes only one mag. ion per structure
ratio of 1.1986570869544226 in ('Fe', 4, 'Fe', 4)
ratio of -1.193820026016113 in ('Fe', 3, 'Fe', 3)
FM to AFM ratio  :
('Fe', 3, 'Fe', 3) 0.06 -1.19 32
('V', 4, 'V', 4) 3.31 0.52 36
('Mn', 2, 'Mn', 2) 0.16 -0.79 53
('Fe', 1, 'Fe', 1) 1.0 0.0 3
('Cr', 3, 'Cr', 3) 0.78 -0.11 48
('Cr', 4, 'Cr', 4) 1.3 0.12 11
('Fe', 2, 'Fe', 2) 0.57 -0.24 32
('Fe', 4, 'Fe', 4) 15.8 1.2 16
('V', 3, 'V', 3) 3.21 0.51 52
('Mn', 4, 'Mn', 4) 2.39 0.38 41
('Co', 2, 'Co', 2) 0.23 -0.64 29
('Co', 4, 'Co', 4) 1.89 0.28 5
('Mn', 3, 'Mn', 3) 4.64 0.67 75
('Ni', 2, 'Ni', 2) 0.28 -0.55 34
('Co', 3, 'Co', 3) 0.2 -0.7 9
('Ni', 3, 'Ni', 3) 2.2 0.34 7
('Cr', 2, 'Cr', 2) 1.57 0.2 5
('V', 2, 'V', 2) 1.06 0.02 8
('Mn', 5, 'Mn', 5) -- (only FM) 1
9 19 0.47368421052631576
Num structures:  497
missed:  set()
